In [1]:
#!/usr/bin/env python3
"""Generate fictional food-rescue data and save it in a SQLite database.
Run with: python community_food_rescue_rubric_aligned.py
"""

from __future__ import annotations

import random
import sqlite3
from datetime import date, datetime, time, timedelta, timezone
from pathlib import Path

OUTPUT_DB = Path("community_food_rescue.sqlite")
BATCH_COUNT = 1_600                 # Above the assignment minimum of 1,000.
RANDOM_SEED = 20261102              # Same seed and settings recreate the same data.
MISSING_RATE = 0.08                 # 8% of original forms miss one optional value.
DUPLICATE_RATE = 0.02              # About 2% of batches get a repeat form.

FOOD_CATEGORIES = ("produce", "bakery", "dairy", "prepared_meal", "pantry")
QUALITY_GRADES = (1, 2, 3, 4, 5)   # Ordered from poor (1) to excellent (5).
AREAS = ("N1", "M1", "LS2", "BS1", "CF10", "EH3", "B1", "G2")

SCHEMA = """
CREATE TABLE donors (
    donor_id INTEGER PRIMARY KEY,
    donor_name TEXT NOT NULL UNIQUE,
    donor_type TEXT NOT NULL CHECK (donor_type IN ('cafe','grocery','bakery','farm','caterer')),
    postcode_area TEXT NOT NULL,
    contact_email TEXT UNIQUE,
    joined_on TEXT NOT NULL,
    is_active INTEGER NOT NULL CHECK (is_active IN (0,1))
);
CREATE TABLE charities (
    charity_id INTEGER PRIMARY KEY,
    charity_name TEXT NOT NULL UNIQUE,
    service_type TEXT NOT NULL CHECK (service_type IN ('community_kitchen','shelter','food_bank','youth_service')),
    priority_level INTEGER NOT NULL CHECK (priority_level BETWEEN 1 AND 4),
    daily_capacity_kg REAL NOT NULL CHECK (daily_capacity_kg > 0),
    postcode_area TEXT NOT NULL,
    is_approved INTEGER NOT NULL CHECK (is_approved IN (0,1))
);
CREATE TABLE volunteers (
    volunteer_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    role TEXT NOT NULL CHECK (role IN ('driver','sorter','coordinator')),
    experience_level INTEGER NOT NULL CHECK (experience_level BETWEEN 1 AND 5),
    joined_on TEXT NOT NULL,
    phone_number TEXT UNIQUE
);
CREATE TABLE food_batches (
    batch_id INTEGER PRIMARY KEY,
    donor_id INTEGER NOT NULL REFERENCES donors(donor_id),
    food_category TEXT NOT NULL CHECK (food_category IN ('produce','bakery','dairy','prepared_meal','pantry')),
    storage_type TEXT NOT NULL CHECK (storage_type IN ('ambient','chilled','frozen')),
    quality_grade INTEGER NOT NULL CHECK (quality_grade BETWEEN 1 AND 5),
    quantity_kg REAL NOT NULL CHECK (quantity_kg > 0 AND quantity_kg <= 500),
    prepared_at TEXT NOT NULL,
    expires_at TEXT NOT NULL CHECK (expires_at > prepared_at),
    storage_temp_c REAL NOT NULL CHECK (storage_temp_c BETWEEN -30 AND 45),
    batch_status TEXT NOT NULL CHECK (batch_status IN ('available','collected','delivered','expired','wasted'))
);
-- This is the raw intake table. Optional reported values can be SQL NULL.
CREATE TABLE intake_submissions (
    intake_id INTEGER PRIMARY KEY,
    source_reference TEXT NOT NULL,
    submitted_at TEXT NOT NULL,
    donor_id INTEGER NOT NULL REFERENCES donors(donor_id),
    batch_id INTEGER NOT NULL REFERENCES food_batches(batch_id),
    reported_category TEXT CHECK (reported_category IS NULL OR reported_category IN ('produce','bakery','dairy','prepared_meal','pantry')),
    reported_quantity_kg REAL CHECK (reported_quantity_kg IS NULL OR reported_quantity_kg > 0),
    reported_expires_at TEXT,
    reported_quality_grade INTEGER CHECK (reported_quality_grade IS NULL OR reported_quality_grade BETWEEN 1 AND 5),
    observed_temp_c REAL CHECK (observed_temp_c IS NULL OR observed_temp_c BETWEEN -30 AND 45),
    review_status TEXT NOT NULL CHECK (review_status IN ('accepted','needs_review','duplicate_flagged')),
    duplicate_of_intake_id INTEGER REFERENCES intake_submissions(intake_id)
);
CREATE TABLE pickups (
    pickup_id INTEGER PRIMARY KEY,
    donor_id INTEGER NOT NULL REFERENCES donors(donor_id),
    volunteer_id INTEGER NOT NULL REFERENCES volunteers(volunteer_id),
    pickup_at TEXT NOT NULL,
    pickup_status TEXT NOT NULL CHECK (pickup_status IN ('completed','cancelled')),
    distance_km REAL NOT NULL CHECK (distance_km > 0 AND distance_km <= 100)
);
-- A batch can be split over separate pickup events. The pair is the key:
-- the same batch cannot be listed twice on one pickup, but can appear on another.
CREATE TABLE pickup_batches (
    pickup_id INTEGER NOT NULL REFERENCES pickups(pickup_id),
    batch_id INTEGER NOT NULL REFERENCES food_batches(batch_id),
    collected_kg REAL NOT NULL CHECK (collected_kg > 0),
    PRIMARY KEY (pickup_id, batch_id)
);
CREATE TABLE deliveries (
    delivery_id INTEGER PRIMARY KEY,
    charity_id INTEGER NOT NULL REFERENCES charities(charity_id),
    volunteer_id INTEGER NOT NULL REFERENCES volunteers(volunteer_id),
    delivered_at TEXT NOT NULL,
    delivery_status TEXT NOT NULL CHECK (delivery_status IN ('completed','partial')),
    distance_km REAL NOT NULL CHECK (distance_km > 0 AND distance_km <= 100)
);
-- The same batch may be divided among deliveries to different organisations.
CREATE TABLE delivery_batches (
    delivery_id INTEGER NOT NULL REFERENCES deliveries(delivery_id),
    batch_id INTEGER NOT NULL REFERENCES food_batches(batch_id),
    delivered_kg REAL NOT NULL CHECK (delivered_kg > 0),
    PRIMARY KEY (delivery_id, batch_id)
);
CREATE TABLE data_quality_notes (
    note_id INTEGER PRIMARY KEY,
    anomaly_type TEXT NOT NULL UNIQUE,
    explanation TEXT NOT NULL,
    affected_rows INTEGER NOT NULL CHECK (affected_rows >= 0),
    rate_percent REAL NOT NULL CHECK (rate_percent BETWEEN 0 AND 100)
);
CREATE INDEX idx_batches_donor ON food_batches(donor_id);
CREATE INDEX idx_batches_status_expiry ON food_batches(batch_status, expires_at);
CREATE INDEX idx_intake_batch ON intake_submissions(batch_id);
"""


def to_iso(value: datetime) -> str:
    return value.astimezone(timezone.utc).replace(microsecond=0).isoformat()


def random_day(rng: random.Random, start: date, end: date) -> date:
    return start + timedelta(days=rng.randint(0, (end - start).days))


def make_reference_data(rng: random.Random):
    """Create the fictional donors, charities, and volunteers used by the FKs."""
    donor_types = ("cafe", "grocery", "bakery", "farm", "caterer")
    donors = []
    for i in range(1, 41):
        donors.append((
            i, f"Fictional Donor {i:02d}", rng.choice(donor_types), rng.choice(AREAS),
            f"donor{i:02d}@example.test" if i % 9 else None,
            random_day(rng, date(2023, 1, 1), date(2026, 1, 1)).isoformat(),
            int(rng.random() >= 0.05),
        ))

    service_types = ("community_kitchen", "shelter", "food_bank", "youth_service")
    charities = []
    for i in range(1, 26):
        charities.append((
            i, f"Fictional Community Partner {i:02d}", rng.choice(service_types),
            rng.choices((1, 2, 3, 4), weights=(0.10, 0.25, 0.40, 0.25))[0],
            round(rng.uniform(35, 420), 1), rng.choice(AREAS), int(rng.random() >= 0.04),
        ))

    first_names = ("Alex", "Sam", "Jordan", "Taylor", "Morgan", "Casey", "Riley", "Jamie")
    last_names = ("Patel", "Reed", "Khan", "Brooks", "Singh", "Clark", "Ali", "Shaw")
    volunteers = []
    for i in range(1, 61):
        volunteers.append((
            i, rng.choice(first_names), rng.choice(last_names),
            rng.choices(("driver", "sorter", "coordinator"), weights=(0.55, 0.30, 0.15))[0],
            rng.choices((1, 2, 3, 4, 5), weights=(0.15, 0.25, 0.30, 0.20, 0.10))[0],
            random_day(rng, date(2022, 1, 1), date(2026, 1, 1)).isoformat(),
            f"000000{i:04d}",
        ))
    return donors, charities, volunteers


def make_batches(rng: random.Random, donors, count: int):
    """Make batches with plausible food types, dates, and amounts."""
    shelf_life = {"produce": 6, "bakery": 3, "dairy": 5, "prepared_meal": 2, "pantry": 90}
    normal_storage = {"produce": "chilled", "bakery": "ambient", "dairy": "chilled",
                     "prepared_meal": "chilled", "pantry": "ambient"}
    temperatures = {"ambient": (15, 25), "chilled": (1, 7), "frozen": (-22, -14)}
    batches = []

    for batch_id in range(1, count + 1):
        category = rng.choices(FOOD_CATEGORIES, weights=(0.30, 0.18, 0.17, 0.15, 0.20))[0]
        storage = normal_storage[category]
        if category in ("produce", "bakery") and rng.random() < 0.12:
            storage = "frozen"

        prepared_day = random_day(rng, date(2026, 1, 1), date(2026, 10, 25))
        days_to_expiry = max(1, shelf_life[category] + rng.randint(-2, 2))
        prepared_at = datetime.combine(
            prepared_day, time(rng.randrange(6, 19), rng.choice((0, 15, 30, 45))),
            tzinfo=timezone.utc,
        )
        expires_at = prepared_at + timedelta(days=days_to_expiry)
        low, high = temperatures[storage]

        batches.append({
            "batch_id": batch_id,
            "donor_id": rng.choice(donors)[0],
            "food_category": category,
            "storage_type": storage,
            "quality_grade": rng.choices(QUALITY_GRADES, weights=(0.04, 0.12, 0.28, 0.36, 0.20))[0],
            "quantity_kg": round(rng.uniform(3, 145), 2),
            "prepared_at": to_iso(prepared_at),
            "expires_at": to_iso(expires_at),
            "storage_temp_c": round(rng.uniform(low, high), 1),
            "batch_status": "available",
        })
    return batches


def make_intake_data(rng: random.Random, batches):
    """Add some missing fields and repeat submissions."""
    forms = []
    original_id = {}
    missing_form_count = 0
    missing_field_count = 0

    for intake_id, batch in enumerate(batches, start=1):
        reported = {
            "category": batch["food_category"],
            "quantity": batch["quantity_kg"],
            "expiry": batch["expires_at"],
            "quality": batch["quality_grade"],
            "temperature": batch["storage_temp_c"],
        }

        # A small share of original forms is missing one optional value.
        if rng.random() < MISSING_RATE:
            field = rng.choice(("quantity", "expiry", "temperature"))
            reported[field] = None
            missing_form_count += 1
            missing_field_count += 1

        submitted_at = to_iso(
            datetime.fromisoformat(batch["prepared_at"])
            - timedelta(minutes=rng.randint(5, 180))
        )
        original_id[batch["batch_id"]] = intake_id
        status = "needs_review" if None in reported.values() else "accepted"

        forms.append((
            intake_id, f"INT-{batch['batch_id']:05d}", submitted_at,
            batch["donor_id"], batch["batch_id"], reported["category"],
            reported["quantity"], reported["expiry"], reported["quality"],
            reported["temperature"], status, None,
        ))

    duplicate_count = max(1, round(len(batches) * DUPLICATE_RATE))
    for batch in rng.sample(batches, duplicate_count):
        original = forms[original_id[batch["batch_id"]] - 1]
        duplicate_id = len(forms) + 1
        duplicate_time = to_iso(
            datetime.fromisoformat(original[2]) + timedelta(minutes=rng.randint(1, 30))
        )
        # A duplicate has the same source reference and batch, but its own ID.
        forms.append((
            duplicate_id, original[1], duplicate_time, original[3], original[4],
            *original[5:10], "duplicate_flagged", original[0],
        ))
    return forms, missing_form_count, missing_field_count, duplicate_count


def make_pickups_and_deliveries(rng, batches, charities, volunteers):
    """Create pickup and delivery events for the batches."""
    by_donor = {}
    for batch in batches:
        by_donor.setdefault(batch["donor_id"], []).append(batch)

    volunteer_ids = [row[0] for row in volunteers]
    charity_ids = [row[0] for row in charities]
    pickup_rows, pickup_links, collected_portions = [], [], []
    next_pickup_id = 1

    for donor_id, donor_batches in by_donor.items():
        donor_batches.sort(key=lambda row: row["prepared_at"])
        for start in range(0, len(donor_batches), 3):
            group = donor_batches[start:start + 3]
            latest_prep = max(datetime.fromisoformat(b["prepared_at"]) for b in group)
            pickup_time = latest_prep + timedelta(hours=rng.randint(2, 20))
            status = "cancelled" if rng.random() < 0.04 else "completed"
            main_pickup_id = next_pickup_id
            next_pickup_id += 1
            pickup_rows.append((
                main_pickup_id, donor_id, rng.choice(volunteer_ids), to_iso(pickup_time),
                status, round(rng.uniform(1.2, 24), 1),
            ))

            if status == "cancelled":
                continue

            for batch in group:
                if datetime.fromisoformat(batch["expires_at"]) <= pickup_time:
                    batch["batch_status"] = "expired"
                    continue

                total_collected = round(batch["quantity_kg"] * rng.uniform(0.82, 1.0), 2)

                # Some batches are collected in two parts at separate pickups.
                # The two rows demonstrate why both columns form the primary key.
                if batch["batch_id"] % 10 == 0 and total_collected >= 1:
                    first_part = round(total_collected * 0.60, 2)
                    second_part = round(total_collected - first_part, 2)
                    pickup_links.append((main_pickup_id, batch["batch_id"], first_part))
                    collected_portions.append((batch, first_part, pickup_time, 0))

                    second_pickup_id = next_pickup_id
                    next_pickup_id += 1
                    second_time = pickup_time + timedelta(minutes=30)
                    pickup_rows.append((
                        second_pickup_id, donor_id, rng.choice(volunteer_ids),
                        to_iso(second_time), "completed", round(rng.uniform(1.2, 24), 1),
                    ))
                    pickup_links.append((second_pickup_id, batch["batch_id"], second_part))
                    collected_portions.append((batch, second_part, second_time, 1))
                else:
                    pickup_links.append((main_pickup_id, batch["batch_id"], total_collected))
                    collected_portions.append((batch, total_collected, pickup_time, 0))

                batch["batch_status"] = "collected"

    # Split portions go to different charities, so a batch can appear on more
    # than one delivery. Other batches go to a randomly chosen charity.
    planned = {}
    for batch, amount, pickup_time, part_number in collected_portions:
        if rng.random() < 0.03:
            batch["batch_status"] = "wasted"
            continue

        if batch["batch_id"] % 10 == 0:
            charity_id = charity_ids[(batch["batch_id"] + part_number * 7) % len(charity_ids)]
        else:
            charity_id = rng.choice(charity_ids)

        planned.setdefault((charity_id, pickup_time.date()), []).append(
            (batch, amount, pickup_time)
        )

    delivery_rows, delivery_links = [], []
    next_delivery_id = 1

    for (charity_id, _day), portions in planned.items():
        for start in range(0, len(portions), 4):
            group = portions[start:start + 4]
            delivery_time = max(item[2] for item in group) + timedelta(hours=2)
            deliverable = []

            for batch, collected_kg, _pickup_time in group:
                if datetime.fromisoformat(batch["expires_at"]) > delivery_time:
                    delivered_kg = round(collected_kg * rng.uniform(0.84, 1.0), 2)
                    deliverable.append((batch, delivered_kg))
                else:
                    batch["batch_status"] = "wasted"

            if not deliverable:
                continue

            partial = any(kg < batch["quantity_kg"] * 0.90 for batch, kg in deliverable)
            delivery_rows.append((
                next_delivery_id, charity_id, rng.choice(volunteer_ids),
                to_iso(delivery_time), "partial" if partial else "completed",
                round(rng.uniform(1, 30), 1),
            ))
            for batch, delivered_kg in deliverable:
                delivery_links.append((next_delivery_id, batch["batch_id"], delivered_kg))
                batch["batch_status"] = "delivered"
            next_delivery_id += 1

    cutoff = datetime(2026, 11, 1, tzinfo=timezone.utc)
    for batch in batches:
        if (batch["batch_status"] == "available"
                and datetime.fromisoformat(batch["expires_at"]) < cutoff):
            batch["batch_status"] = "expired"

    return pickup_rows, pickup_links, delivery_rows, delivery_links


def insert_rows(conn, table, columns, rows):
    if rows:
        marks = ",".join("?" for _ in rows[0])
        conn.executemany(f"INSERT INTO {table} ({columns}) VALUES ({marks})", rows)


def validate_database(conn, expected_batch_count):
    """Stop with an error if a key size, range, or relationship check fails."""
    if conn.execute("PRAGMA foreign_keys").fetchone()[0] != 1:
        raise RuntimeError("Foreign-key enforcement is not enabled.")

    batch_count = conn.execute("SELECT COUNT(*) FROM food_batches").fetchone()[0]
    if batch_count < 1_000 or batch_count != expected_batch_count:
        raise RuntimeError(f"Unexpected food_batches row count: {batch_count}.")

    if conn.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
        raise RuntimeError("SQLite integrity_check failed.")
    if conn.execute("PRAGMA foreign_key_check").fetchall():
        raise RuntimeError("SQLite foreign_key_check found invalid references.")

    invalid_batches = conn.execute("""
        SELECT COUNT(*) FROM food_batches
        WHERE quantity_kg <= 0 OR quantity_kg > 500
           OR expires_at <= prepared_at
           OR quality_grade NOT BETWEEN 1 AND 5
           OR storage_temp_c NOT BETWEEN -30 AND 45
    """).fetchone()[0]
    if invalid_batches:
        raise RuntimeError(f"Found {invalid_batches} food batches outside valid ranges.")

    too_much_collected = conn.execute("""
        SELECT COUNT(*) FROM (
            SELECT b.batch_id
            FROM food_batches AS b
            JOIN pickup_batches AS pb USING (batch_id)
            GROUP BY b.batch_id, b.quantity_kg
            HAVING SUM(pb.collected_kg) > b.quantity_kg + 0.01
        )
    """).fetchone()[0]
    if too_much_collected:
        raise RuntimeError("A batch has more collected food than its original quantity.")

    too_much_delivered = conn.execute("""
        SELECT COUNT(*) FROM (
            SELECT db.batch_id
            FROM delivery_batches AS db
            GROUP BY db.batch_id
            HAVING SUM(db.delivered_kg) > (
                SELECT COALESCE(SUM(pb.collected_kg), 0)
                FROM pickup_batches AS pb
                WHERE pb.batch_id = db.batch_id
            ) + 0.01
        )
    """).fetchone()[0]
    if too_much_delivered:
        raise RuntimeError("A batch has more delivered food than was collected.")

    delivered_before_pickup = conn.execute("""
        SELECT COUNT(*) FROM delivery_batches AS db
        JOIN pickup_batches AS pb USING (batch_id)
        JOIN deliveries AS d USING (delivery_id)
        JOIN pickups AS p USING (pickup_id)
        WHERE d.delivered_at < p.pickup_at
    """).fetchone()[0]
    if delivered_before_pickup:
        raise RuntimeError("Found a delivery dated before its pickup.")


def print_summary(path, seed):
    """Print row counts, columns, keys, constraints, and data-quality totals."""
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA foreign_keys = ON")
    try:
        print(f"\nDatabase created: {path.resolve()}")
        print(f"Random seed: {seed}")
        print("\nTABLE COUNTS AND COLUMNS")

        tables = conn.execute("""
            SELECT name FROM sqlite_master
            WHERE type='table' AND name NOT LIKE 'sqlite_%'
            ORDER BY name
        """).fetchall()

        for (table,) in tables:
            count = conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
            print(f"\n{table}: {count:,} rows")
            for col in conn.execute(f'PRAGMA table_info("{table}")'):
                pk = " PK" if col[5] else ""
                required = " NOT NULL" if col[3] else " NULL allowed"
                print(f"  {col[1]} {col[2]}{pk}{required}")

            foreign_keys = conn.execute(f'PRAGMA foreign_key_list("{table}")').fetchall()
            for fk in foreign_keys:
                print(f"  FK: {fk[3]} -> {fk[2]}.{fk[4]}")

        print("\nTABLE DEFINITIONS (includes UNIQUE and CHECK constraints)")
        for (table,) in tables:
            ddl = conn.execute(
                "SELECT sql FROM sqlite_master WHERE type='table' AND name=?",
                (table,),
            ).fetchone()[0]
            print(f"\n{ddl}")

        original_forms = conn.execute("""
            SELECT COUNT(*) FROM intake_submissions
            WHERE duplicate_of_intake_id IS NULL
        """).fetchone()[0]
        missing_fields = conn.execute("""
            SELECT SUM(reported_quantity_kg IS NULL) +
                   SUM(reported_expires_at IS NULL) +
                   SUM(observed_temp_c IS NULL)
            FROM intake_submissions
            WHERE duplicate_of_intake_id IS NULL
        """).fetchone()[0]
        forms_with_missing = conn.execute("""
            SELECT COUNT(*) FROM intake_submissions
            WHERE duplicate_of_intake_id IS NULL
              AND (reported_quantity_kg IS NULL OR reported_expires_at IS NULL
                   OR observed_temp_c IS NULL)
        """).fetchone()[0]
        duplicate_rows = conn.execute("""
            SELECT COUNT(*) FROM intake_submissions
            WHERE duplicate_of_intake_id IS NOT NULL
        """).fetchone()[0]
        duplicate_groups = conn.execute("""
            SELECT COUNT(*) FROM (
                SELECT source_reference, batch_id
                FROM intake_submissions
                GROUP BY source_reference, batch_id
                HAVING COUNT(*) > 1
            )
        """).fetchone()[0]

        print("\nMISSING AND DUPLICATE DATA")
        print(f"Original intake forms: {original_forms:,}")
        print(f"Missing optional fields (SQL NULL): {missing_fields} "
              f"({100 * missing_fields / original_forms:.2f}% of original forms)")
        print(f"Forms with at least one missing field: {forms_with_missing} "
              f"({100 * forms_with_missing / original_forms:.2f}%)")
        print(f"Duplicate submissions: {duplicate_rows} "
              f"({100 * duplicate_rows / original_forms:.2f}%)")
        print(f"Repeated business-key groups: {duplicate_groups}")

        print("\nDATABASE CHECKS")
        print("Integrity check:", conn.execute("PRAGMA integrity_check").fetchone()[0])
        fk_issues = conn.execute("PRAGMA foreign_key_check").fetchall()
        print(f"Foreign-key violations: {len(fk_issues)}")
        print("Foreign keys enabled for this connection:",
              conn.execute("PRAGMA foreign_keys").fetchone()[0])
        print("Quantity, date, and delivery checks passed during generation.")
        print("\nMeasurement examples:")
        print("  Nominal: food_category")
        print("  Ordinal: quality_grade (1 poor to 5 excellent)")
        print("  Interval: storage_temp_c in Celsius")
        print("  Ratio: quantity_kg")
    finally:
        conn.close()


def create_database(path=OUTPUT_DB, batch_count=BATCH_COUNT, seed=RANDOM_SEED):
    """Create the database from scratch and print the checks at the end."""
    if batch_count < 1_000:
        raise ValueError("BATCH_COUNT must be at least 1,000.")

    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()  # Re-running the script replaces the old generated file.

    rng = random.Random(seed)
    donors, charities, volunteers = make_reference_data(rng)
    batches = make_batches(rng, donors, batch_count)
    forms, missing_forms, missing_fields, duplicate_count = make_intake_data(rng, batches)
    pickups, pickup_links, deliveries, delivery_links = make_pickups_and_deliveries(
        rng, batches, charities, volunteers
    )

    conn = sqlite3.connect(path)
    try:
        # SQLite foreign-key checks must be enabled on each database connection.
        conn.execute("PRAGMA foreign_keys = ON")
        conn.executescript(SCHEMA)

        with conn:
            insert_rows(conn, "donors",
                "donor_id,donor_name,donor_type,postcode_area,contact_email,joined_on,is_active", donors)
            insert_rows(conn, "charities",
                "charity_id,charity_name,service_type,priority_level,daily_capacity_kg,postcode_area,is_approved", charities)
            insert_rows(conn, "volunteers",
                "volunteer_id,first_name,last_name,role,experience_level,joined_on,phone_number", volunteers)

            batch_columns = (
                "batch_id,donor_id,food_category,storage_type,quality_grade,quantity_kg,"
                "prepared_at,expires_at,storage_temp_c,batch_status"
            )
            batch_rows = [tuple(b[column] for column in batch_columns.split(",")) for b in batches]
            insert_rows(conn, "food_batches", batch_columns, batch_rows)

            intake_columns = (
                "intake_id,source_reference,submitted_at,donor_id,batch_id,reported_category,"
                "reported_quantity_kg,reported_expires_at,reported_quality_grade,observed_temp_c,"
                "review_status,duplicate_of_intake_id"
            )
            insert_rows(conn, "intake_submissions", intake_columns, forms)
            insert_rows(conn, "pickups",
                "pickup_id,donor_id,volunteer_id,pickup_at,pickup_status,distance_km", pickups)
            insert_rows(conn, "pickup_batches", "pickup_id,batch_id,collected_kg", pickup_links)
            insert_rows(conn, "deliveries",
                "delivery_id,charity_id,volunteer_id,delivered_at,delivery_status,distance_km", deliveries)
            insert_rows(conn, "delivery_batches", "delivery_id,batch_id,delivered_kg", delivery_links)

            notes = [
                (1, "missing_intake_fields",
                 "Some original forms omit one optional quantity, expiry, or temperature value. "
                 "The missing value is stored as SQL NULL and the form is flagged for review.",
                 missing_forms, round(100 * missing_forms / batch_count, 2)),
                (2, "duplicate_intake_submissions",
                 "A duplicate repeats the same source reference and batch, but has its own "
                 "intake ID and points to the original form.",
                 duplicate_count, round(100 * duplicate_count / batch_count, 2)),
            ]
            insert_rows(conn, "data_quality_notes",
                "note_id,anomaly_type,explanation,affected_rows,rate_percent", notes)

        validate_database(conn, batch_count)
    finally:
        conn.close()

    print_summary(path, seed)


if __name__ == "__main__":
    create_database()


Database created: /content/community_food_rescue.sqlite
Random seed: 20261102

TABLE COUNTS AND COLUMNS

charities: 25 rows
  charity_id INTEGER PK NULL allowed
  charity_name TEXT NOT NULL
  service_type TEXT NOT NULL
  priority_level INTEGER NOT NULL
  daily_capacity_kg REAL NOT NULL
  postcode_area TEXT NOT NULL
  is_approved INTEGER NOT NULL

data_quality_notes: 2 rows
  note_id INTEGER PK NULL allowed
  anomaly_type TEXT NOT NULL
  explanation TEXT NOT NULL
  affected_rows INTEGER NOT NULL
  rate_percent REAL NOT NULL

deliveries: 910 rows
  delivery_id INTEGER PK NULL allowed
  charity_id INTEGER NOT NULL
  volunteer_id INTEGER NOT NULL
  delivered_at TEXT NOT NULL
  delivery_status TEXT NOT NULL
  distance_km REAL NOT NULL
  FK: volunteer_id -> volunteers.volunteer_id
  FK: charity_id -> charities.charity_id

delivery_batches: 996 rows
  delivery_id INTEGER PK NOT NULL
  batch_id INTEGER PK NOT NULL
  delivered_kg REAL NOT NULL
  FK: batch_id -> food_batches.batch_id
  FK: deli